# Langchain Tools

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(model="gpt-5.4-mini")   # Initialize the OpenAI model with GPT-5

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import HumanMessage

@tool("calculator", description="A simple calculator that can evaluate mathematical expressions.")
def calculator_tool(a: int, b: int, operation: str) -> int:
    if operation == "add":
        return "Addition result: " + str(a + b)
    elif operation == "subtract":
        return "Subtraction result: " + str(a - b)
    elif operation == "multiply":
        return "Multiplication result: " + str(a * b)
    elif operation == "divide":
        return "Division result: " + str(a / b if b != 0 else float('inf'))

agent = create_agent(model=llm, tools=[calculator_tool], system_prompt="You need to use tools to answer the user's question.")  # Create an agent using the initialized LLM

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="What is 5 plus 3?")]},
    stream_mode="messages"
):
    if token.content:
        print(token.content, end="", flush=True)

In [ ]:
from tavily import TavilyClient
from langchain.messages import HumanMessage

@tool("web_search", description="A tool to perform web searches for finding latest information")
def web_search(query: str):
    tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
    search_results = tavily_client.search(query)
    return search_results

agent = create_agent(model=llm, tools=[web_search], system_prompt="You are a helpful chat assistant that can perform web searches to find the latest information for the user.")

response = agent.invoke({"messages": [HumanMessage(content="What is the latest news about AI? Give me a summary of the top 3 articles you find along with their sources.")]})

print(response["messages"][-1].content)